# Hands-on 29 (Bab 11): Mini Proyek Klasifikasi Sederhana di Raspberry Pi / Emulator

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 11.

Mini proyek end-to-end meniru deployment edge: simulator sensor -> model lokal
(joblib dari Hands-on 27) -> status klasifikasi -> log CSV dan/atau MQTT.

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

# --- Utilitas notebook: menjalankan skrip buku APA ADANYA (tanpa flag) ---
# Skrip buku memakai while True / loop_forever, jadi notebook menghentikannya
# dari luar setelah beberapa detik dan menyimpan keluaran parsialnya.
import subprocess, sys, time
from pathlib import Path

LOG_DIR = Path("outputs/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

def latar(script, nama_log, args=()):
    """Jalankan skrip buku di latar belakang (python -u), keluaran ke file log."""
    f = open(LOG_DIR / nama_log, "w", encoding="utf-8")
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=f, stderr=subprocess.STDOUT)
    return p, f

def henti(*pasangan):
    """Hentikan proses latar belakang dan tutup file lognya."""
    for p, f in pasangan:
        p.terminate()
        try:
            p.wait(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
        f.close()

def jalankan_buku(script, detik, args=()):
    """Jalankan skrip buku selama `detik` detik lalu hentikan; kembalikan keluaran."""
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        out, _ = p.communicate(timeout=detik)
    except subprocess.TimeoutExpired:
        p.terminate()
        try:
            out, _ = p.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
            out, _ = p.communicate()
    return out

def log_teks(nama_log):
    return (LOG_DIR / nama_log).read_text(encoding="utf-8")


[INFO] Working directory: /home/sandbox/iot-ml-book


## 1. Mini Proyek tanpa MQTT (simulator -> model -> log CSV)

In [2]:
print(jalankan_buku("scripts/edge_room_classifier_project.py", 90))

=== Mini proyek edge room classifier dimulai ===
step=0 | T=30.05 H=66.59 CO2=740 P=151.38 -> status=warning
step=1 | T=24.66 H=61.88 CO2=814 P=167.53 -> status=normal
step=2 | T=32.38 H=58.93 CO2=561 P=220.85 -> status=warning
step=3 | T=32.19 H=51.16 CO2=919 P=252.07 -> status=warning
step=4 | T=25.87 H=50.24 CO2=590 P=244.05 -> status=normal
step=5 | T=32.01 H=54.71 CO2=1281 P=101.89 -> status=warning
step=6 | T=25.25 H=67.81 CO2=928 P=245.12 -> status=normal
step=7 | T=29.2 H=49.04 CO2=559 P=159.48 -> status=warning
step=8 | T=28.67 H=61.61 CO2=580 P=102.18 -> status=normal
step=9 | T=26.97 H=59.04 CO2=1170 P=181.37 -> status=normal
step=10 | T=27.71 H=59.95 CO2=830 P=294.65 -> status=normal
step=11 | T=29.75 H=69.02 CO2=596 P=262.34 -> status=warning
step=12 | T=29.32 H=43.66 CO2=713 P=174.12 -> status=warning
step=13 | T=27.19 H=54.26 CO2=1455 P=187.34 -> status=normal
step=14 | T=29.2 H=52.69 CO2=1258 P=227.86 -> status=warning
step=15 | T=28.7 H=59.54 CO2=520 P=102.26 -> status

## 2. Verifikasi Log CSV

In [3]:
import pandas as pd
log = pd.read_csv("outputs/reports/edge_room_predictions.csv")
print(log.head(10).to_string(index=False))
print("\nDistribusi status:", log["predicted_status"].value_counts().to_dict())

 step  temperature_c  humidity_pct  co2_ppm  power_watt predicted_status
    0          30.05         66.59      740      151.38          warning
    1          24.66         61.88      814      167.53           normal
    2          32.38         58.93      561      220.85          warning
    3          32.19         51.16      919      252.07          warning
    4          25.87         50.24      590      244.05           normal
    5          32.01         54.71     1281      101.89          warning
    6          25.25         67.81      928      245.12           normal
    7          29.20         49.04      559      159.48          warning
    8          28.67         61.61      580      102.18           normal
    9          26.97         59.04     1170      181.37           normal

Distribusi status: {'normal': 15, 'warning': 12, 'fault': 3}


## 3. Versi dengan MQTT (broker lokal)

In [4]:
import json
import paho.mqtt.subscribe as mqttsub
broker = subprocess.Popen([sys.executable, "scripts/mqtt/run_broker.py"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
try:
    # Subscriber penerima prediksi (skrip buku scripts/mqtt/subscribe_sensor.py memakai topik room01/telemetry,
    # jadi di sini dipakai paho.mqtt.subscribe untuk topik prediksi), lalu skrip edge buku dijalankan apa adanya.
    import threading
    hasil = {}
    def dengar():
        hasil["msgs"] = mqttsub.simple("iotlab/room01/prediction", hostname="127.0.0.1", port=1883, qos=1, msg_count=6)
    t = threading.Thread(target=dengar); t.start()
    time.sleep(1)
    out = jalankan_buku("scripts/edge_room_classifier_mqtt.py", 30)
    print(out)
    ringkas = __import__("re").findall(r"'predicted_status': '(\w+)'", out)
    print("Jumlah publish:", len(ringkas), "| distribusi:", dict(__import__("collections").Counter(ringkas)))
    t.join(timeout=10)
    print("--- Pesan prediksi yang diterima subscriber ---")
    for m in hasil.get("msgs", []):
        print(m.topic, m.payload.decode())
finally:
    broker.terminate()

Published: {'step': 0, 'temperature_c': 29.07, 'humidity_pct': 57.95, 'co2_ppm': 1474, 'power_watt': 171.97, 'predicted_status': 'warning'}
Published: {'step': 1, 'temperature_c': 31.84, 'humidity_pct': 52.16, 'co2_ppm': 825, 'power_watt': 298.68, 'predicted_status': 'warning'}
Published: {'step': 2, 'temperature_c': 31.67, 'humidity_pct': 58.69, 'co2_ppm': 556, 'power_watt': 288.67, 'predicted_status': 'warning'}
Published: {'step': 3, 'temperature_c': 25.61, 'humidity_pct': 42.64, 'co2_ppm': 518, 'power_watt': 257.29, 'predicted_status': 'normal'}
Published: {'step': 4, 'temperature_c': 31.92, 'humidity_pct': 56.93, 'co2_ppm': 657, 'power_watt': 244.18, 'predicted_status': 'warning'}
Published: {'step': 5, 'temperature_c': 24.34, 'humidity_pct': 57.79, 'co2_ppm': 1033, 'power_watt': 218.7, 'predicted_status': 'normal'}
Published: {'step': 6, 'temperature_c': 25.24, 'humidity_pct': 53.7, 'co2_ppm': 1497, 'power_watt': 145.2, 'predicted_status': 'warning'}
Published: {'step': 7, 'tempe

## Penutup

Mini proyek menyatukan seluruh konsep Bab 11: training di laptop, ekspor model,
inferensi lokal per sampel, lalu hasil diteruskan ke log CSV dan/atau broker MQTT.
Prediksi yang diterbitkan ke topik `iotlab/room01/prediction` terverifikasi diterima
subscriber. Log simulasi tersimpan di `outputs/reports/edge_room_predictions.csv`.

## Catatan dari buku: pengamatan, variasi, dan latihan

Bagian berikut disalin dari buku (Hands-on 31 dalam urutan buku) sebagai panduan pengamatan dan latihan mandiri setelah menjalankan notebook ini.

### 8. Langkah kerja yang harus dilakukan pembaca

Tahap 1
Latih model di laptop Ubuntu.
Tahap 2
Ekspor model:
- joblib untuk model klasik
- .tflite untuk model Keras ringan
Tahap 3
Uji inferensi lokal satu per satu.
Tahap 4
Simulasikan aliran data seolah-olah dari sensor.
Tahap 5
Jalankan mini proyek di Raspberry Pi atau emulator Linux.

Tahap 6
Tambahkan output:

terminal
CSV
MQTT
atau dashboard

Tahap 7
Evaluasi apakah model:

cukup cepat
cukup stabil
cukup ringan
dan layak untuk edge